## Question 3 — Self-attention

Project the token matrix $X$ with $W_Q$, $W_K$, and $W_V$, then run scaled dot-product attention.

(a) If $X\in\mathbb R^{n\times d_{model}}$, choose $W_Q,W_K\in\mathbb R^{d_{model}\times d_k}$ and $W_V\in\mathbb R^{d_{model}\times d_v}$. Then $Q,K\in\mathbb R^{n\times d_k}$, $V\in\mathbb R^{n\times d_v}$. $W_Q,W_K,W_V$ can differ because matching and transmitted content serve different roles, and $d_v$ need not equal $d_k$.

### Code Execution

In [4]:
import numpy as np

**Attention from Question 2.** Self-attention calls this scaled dot-product function.

In [5]:
def stable_softmax(x, axis=-1):
    """Numerically stable softmax along one axis."""
    x = np.asarray(x, dtype=float)
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_shifted = np.exp(shifted)
    return exp_shifted / np.sum(exp_shifted, axis=axis, keepdims=True)


def scaled_dot_product_attention(Q, K, V):
    """Q:(nq, dk), K:(nk, dk), V:(nk, dv) -> output (nq, dv), weights (nq, nk)."""
    Q, K, V = (np.asarray(z, dtype=float) for z in (Q, K, V))
    if Q.ndim != 2 or K.ndim != 2 or V.ndim != 2 or Q.shape[1] != K.shape[1] or K.shape[0] != V.shape[0]:
        raise ValueError("Incompatible Q, K, V dimensions")
    scores = (Q @ K.T) / np.sqrt(Q.shape[1])
    weights = stable_softmax(scores, axis=-1)
    return weights @ V, weights

**Self-attention.** Form $Q=XW_Q$, $K=XW_K$, and $V=XW_V$, then return the output and the weights.

In [6]:
def self_attention(X, W_Q, W_K, W_V):
    """Project (n, d_model) X, then return attention output and weights."""
    Q, K, V = X @ W_Q, X @ W_K, X @ W_V
    return scaled_dot_product_attention(Q, K, V)

**Check.** These projections give the same weights as identity projections. The output is about $[[1.786, 1.893], [1.340, 1.670]]$.

In [7]:
X3 = np.array([[1.0, 2.0], [0.0, 1.0]])
WQ3 = np.eye(2)
WK3 = np.array([[0.0, 1.0], [1.0, 0.0]])
WV3 = np.diag([2.0, 1.0])

out3, weights3 = self_attention(X3, WQ3, WK3, WV3)
identity_out3, identity_weights3 = self_attention(X3, np.eye(2), np.eye(2), np.eye(2))

print("weights:\n", np.round(weights3, 6))
print("output:\n", np.round(out3, 6))
print("identity weights:\n", np.round(identity_weights3, 6))
print("identity output:\n", np.round(identity_out3, 6))

np.testing.assert_allclose(weights3, identity_weights3)
np.testing.assert_allclose(
    out3, [[1.785916, 1.892958], [1.339523, 1.669762]], atol=1e-6
)
print("Self-attention check passed.")

weights:
 [[0.892958 0.107042]
 [0.669762 0.330238]]
output:
 [[1.785916 1.892958]
 [1.339523 1.669762]]
identity weights:
 [[0.892958 0.107042]
 [0.669762 0.330238]]
identity output:
 [[0.892958 1.892958]
 [0.669762 1.669762]]
Self-attention check passed.
